# **Physics Generator — Design Philosophy**

The physics generator creates synthetic missile engagement scenarios for ATAS.

Real engagement data is difficult and unrealistic to obtain at scale, so synthetic data is used to generate controlled training scenarios with known ground-truth labels.

The generator uses metadata-derived aircraft capability ranges from `aircraft_metadata.csv` instead of fully hardcoded values.


### **aircraft_metadata.csv**

This CSV is the threat database for ATAS.

The classifier only predicts:
```python
"F22"
````

The metadata adds tactical information:

* missile speed
* missile range
* aircraft generation
* maneuverability
* combat capability

This allows the system to simulate engagement scenarios.

---

### Columns

#### `aircraft`

Aircraft class name.

Used to match classifier output with metadata.

---

#### `missile_speed`

Approx missile speed (m/s).

Used for:

* closure rate
* evasion time
* threat level

Higher speed = less reaction time.

---

#### `missile_range`

Approx max missile range (m).

Used for:

* launch distance generation
* engagement realism

Higher range = longer reach.

---

#### `enemy_generation`

Aircraft technology level.

Values:

* 3.5
* 4
* 4.5
* 5

Used for:

* threat weighting
* hit probability modifiers

Higher generation = more dangerous.

---

#### `maneuverability`

Aircraft agility.

Values:

* 0 = low
* 1 = medium
* 2 = high

Used for:

* evasion logic
* survival probability

Higher maneuverability = harder to hit.

---

#### `no_aa_capability`

Whether aircraft lacks air-to-air combat capability.

Values:

* 0 = combat capable
* 1 = not combat capable

Used to:

* lower threat score
* skip missile logic for support aircraft

---

## Why This Exists

The metadata converts:

```python
"What aircraft is this?"
```

into:

```python
"How dangerous is this aircraft?"
```

In [2]:
# Importing the libraries

import pandas as pd
import numpy as np

In [3]:
# Importing the CSV to work with

df = pd.read_csv("../data/aircraft_metadata.csv")
df

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
0,A10,222,857,35000,4.0,1,0
1,A400M,255,-1,-1,4.0,0,1
2,AG600,155,-1,-1,4.0,0,1
3,AH64,101,750,8000,4.0,1,0
4,AKINCI,100,1372,65000,4.0,1,0
...,...,...,...,...,...,...,...
97,Y20,255,-1,-1,4.0,0,1
98,YF23,648,1372,160000,5.0,2,0
99,Z10,83,686,8000,4.0,1,0
100,Z19,78,686,8000,4.0,1,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   aircraft            102 non-null    str    
 1   aircraft_max_speed  102 non-null    int64  
 2   missile_speed       102 non-null    int64  
 3   missile_range       102 non-null    int64  
 4   enemy_generation    102 non-null    float64
 5   maneuverability     102 non-null    int64  
 6   no_aa_capability    102 non-null    int64  
dtypes: float64(1), int64(5), str(1)
memory usage: 5.7 KB


In [5]:
df.describe(include="all")

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
count,102,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000
unique,102,NaN,NaN,NaN,NaN,NaN,NaN
top,A10,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,345.764706,636.460784,64724.990196,4.117647,0.852941,0.500000
std,NaN,226.619242,692.537831,102267.602554,0.478118,0.825306,0.502469
min,NaN,61.000000,-1.000000,-1.000000,3.500000,0.000000,0.000000
25%,NaN,164.500000,-1.000000,-1.000000,4.000000,0.000000,0.000000
50%,NaN,268.000000,342.500000,3999.500000,4.000000,1.000000,0.500000
75%,NaN,544.000000,1372.000000,108750.000000,4.500000,2.000000,1.000000


In [6]:
print(df["enemy_generation"].unique())
print(df["maneuverability"].unique())
print(df["no_aa_capability"].unique())

[4.  3.5 4.5 5. ]
[1 0 2]
[0 1]


In [7]:
# Getting data of aircarfts that has air-to-air combact ability
combat_df = df[df["no_aa_capability"]==0].reset_index(drop=True)
combat_df.head()

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
0,A10,222,857,35000,4.0,1,0
1,AH64,101,750,8000,4.0,1,0
2,AKINCI,100,1372,65000,4.0,1,0
3,AV8B,300,1372,160000,4.0,1,0
4,EF2000,590,1372,200000,4.5,2,0


In [8]:
combat_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   aircraft            51 non-null     str    
 1   aircraft_max_speed  51 non-null     int64  
 2   missile_speed       51 non-null     int64  
 3   missile_range       51 non-null     int64  
 4   enemy_generation    51 non-null     float64
 5   maneuverability     51 non-null     int64  
 6   no_aa_capability    51 non-null     int64  
dtypes: float64(1), int64(5), str(1)
memory usage: 2.9 KB


In [9]:
combat_df.describe(include="all")

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
count,51,51.000000,51.000000,51.000000,51.000000,51.000000,51.0
unique,51,NaN,NaN,NaN,NaN,NaN,NaN
top,A10,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,440.235294,1273.921569,129450.980392,4.284314,1.529412,0.0
std,NaN,225.509609,373.955497,112160.655085,0.461030,0.504101,0.0
min,NaN,78.000000,686.000000,8000.000000,3.500000,1.000000,0.0
25%,NaN,246.500000,857.000000,37500.000000,4.000000,1.000000,0.0
50%,NaN,532.000000,1372.000000,110000.000000,4.000000,2.000000,0.0
75%,NaN,590.000000,1372.000000,160000.000000,4.500000,2.000000,0.0


In [10]:
# Getting missile speed ranges
print("Missile Speed ranges: \n")
print(f"Max → {combat_df['missile_speed'].max()} \nMin → {combat_df['missile_speed'].min()}")

Missile Speed ranges: 

Max → 2058 
Min → 686


In [11]:
# Getting missile speed ranges
print("Missile range: \n")
print(f"Max → {combat_df['missile_range'].max()} \nMin → {combat_df['missile_range'].min()}")

Missile range: 

Max → 400000 
Min → 8000


In [12]:
# Getting aircraft speed ranges
print("Aircraft speed range: \n")
print(f"Max → {df['aircraft_max_speed'].max()} \nMin → {df['aircraft_max_speed'].min()}")

Aircraft speed range: 

Max → 983 
Min → 61


In [13]:
# Getting the number of unique generation
combat_df["enemy_generation"].value_counts()

enemy_generation
4.0    25
4.5    11
5.0    11
3.5     4
Name: count, dtype: int64

In [14]:
# Getting the maneverability level
combat_df["maneuverability"].value_counts()

maneuverability
2    27
1    24
Name: count, dtype: int64

In [15]:
# get values of ranges of missile speed
combat_df.sort_values("missile_speed", ascending=False).head(10)

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
14,FCK1,532,2058,100000,4.0,2,0
39,Su57,590,2058,400000,5.0,2,0
32,Mig31,833,2058,400000,4.0,1,0
18,J36,590,1715,400000,5.0,2,0
19,J50,590,1715,400000,5.0,2,0
17,J35,590,1715,300000,5.0,2,0
16,J20,590,1715,300000,5.0,2,0
15,J10,650,1715,300000,4.5,2,0
10,F2,590,1544,105000,4.5,2,0
41,Tejas,550,1544,110000,4.5,2,0


In [16]:
# Get sorted decending missile ranges
combat_df.sort_values("missile_range", ascending=False)[:10]

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
19,J50,590,1715,400000,5.0,2,0
39,Su57,590,2058,400000,5.0,2,0
18,J36,590,1715,400000,5.0,2,0
32,Mig31,833,2058,400000,4.0,1,0
17,J35,590,1715,300000,5.0,2,0
16,J20,590,1715,300000,5.0,2,0
15,J10,650,1715,300000,4.5,2,0
34,Rafale,531,1372,200000,4.5,2,0
20,JAS39,590,1372,200000,4.5,2,0
24,KF21,532,1372,200000,4.5,2,0


In [17]:
combat_df[combat_df["aircraft"]=="F22"]

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
11,F22,669,1372,160000,5.0,2,0


In [18]:
# correlation between aircraft speed and missile 
combat_df[["aircraft_max_speed", "missile_speed", "missile_range"]].corr()

,aircraft_max_speed,missile_speed,missile_range
aircraft_max_speed,1.000000,0.753865,0.633663
missile_speed,0.753865,1.000000,0.820434
missile_range,0.633663,0.820434,1.000000


In [19]:
combat_df.groupby("enemy_generation")["missile_range"].mean()

enemy_generation
3.5     26500.000000
4.0     83240.000000
4.5    159090.909091
5.0    242272.727273
Name: missile_range, dtype: float64

## **Extracted Metadata Insights**

### **Combat-Capable Aircraft**
- Total aircraft: 102
- Combat-capable aircraft: 56
- Non combat-capable aircraft: 46

---

### Aircraft Speed Range

```python
61 m/s → 983 m/s
```

---

### Missile Speed Range

```python 
686 m/s → 2058 m/s
````

---

### Missile Range

```python id="e2h15d"
8,000 m → 400,000 m
```

---

### Enemy Generation Distribution

```python id="txut2q"
4.0  → 23 aircraft
5.0  → 13 aircraft
4.5  → 10 aircraft
3.5  → 5 aircraft
```

---

### Maneuverability Distribution

```python id="nt11ao"
2 (high)   → 27 aircraft
1 (medium) → 24 aircraft
```

---

### Key Observations

* Most combat aircraft belong to Gen 4 and Gen 5.
* Most combat aircraft have medium or high maneuverability.
* Advanced aircraft generally have longer missile ranges.
* High-end aircraft like Su57, Mig31, J35, J36, and J50 dominate the upper missile range limits.

The goal of the generator is not perfect aerospace simulation, but generation of believable and internally consistent combat scenarios that allow ML models to learn meaningful relationships.

---

## **Plan to build physics generator**

### **How it should be called**

```python
from src.physics_generator import generate_dataset
generate_dataset()
```

### **How it will be build**

```markdown
generate_dataset()          ← the one function the notebook calls
    └── load_metadata()     ← reads aircraft_metadata.csv
    └── generate_row()      ← builds one scenario
            └── derive_missile_phase()
            └── derive_closure_rate()
            └── derive_evasion_time()
            └── derive_hit_label()
    └── save_dataset()      ← saves to CSV
```
---

### **The sequence of `physics_generator.py` script**

```python
_generate_row()
│
├── 1. Pick a random enemy aircraft from metadata
│      (combat-capable only — no_aa_capability = 0)
│
├── 2. Sample raw feature values
│      your_speed, your_altitude, azimuth, elevation,
│      maneuverability, countermeasure_deployed
│      launch_distance (within that aircraft's missile_range)
│      remaining_distance (between 0 and launch_distance)
│
├── 3. _derive_missile_phase()
│      "How far has the missile already traveled?"
│      → returns 0, 1, or 2
│
├── 4. _derive_closure_rate()
│      "How fast is the gap closing in 3D space?"
│      → returns a speed in m/s
│
├── 5. _derive_evasion_time()
│      "How many seconds before it reaches you?"
│      → returns a float in seconds
│
├── 6. _derive_hit_label()
│      "After your evasion attempt, does it hit?"
│      → returns 0 or 1
│
└── 7. Package everything into a dict → one row
```

---

## **Sanity Check - Physics Generator**

Quick validation that `generate_dataset()` produces clean, physically sensible output.
Run with `N_ROWS = 1_000` before the full 1M generation.

In [26]:
import sys
from pathlib import Path

# Add src/ to path so we can import physics_generator
sys.path.append(str(Path.cwd().resolve().parent.parent / "src"))

from physics_generator import generate_dataset

In [27]:
# Running the generator for N_ROWS = 1_000
df = generate_dataset()

Done. 1,000 rows saved to /mnt/d/Eakem/Learning ML and DS/Project_ATAS/ATAS_Project/data/synthetic_engagements.csv


In [28]:
import pandas as pd
df = pd.read_csv("../data/synthetic_engagements.csv")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"Shape: {df.shape}")

Rows: 1000
Columns: 16
Shape: (1000, 16)


In [ ]:
df.head()

,launch_distance,remaining_distance,closure_rate,azimuth,elevation,missile_phase,your_speed,your_altitude,your_maneuverability,enemy_altitude,missile_speed,missile_range,enemy_generation,countermeasure_deployed,evasion_time,hit
0,22570.893367,2961.980067,1185.763992,61.005549,-34.707563,2,825.050818,6512.661972,1,3655.798699,857,35000,4.0,1,2.229421,0
1,2367.007196,1652.662358,625.878279,212.466382,68.630275,0,751.777144,18782.156933,1,19508.648175,857,35000,4.0,0,2.772577,1
2,6602.346475,2558.752556,871.179381,281.112640,45.266055,1,104.527357,3625.183143,2,29894.396018,857,35000,4.0,0,3.230825,0
3,18281.792405,18049.709650,648.636953,115.135435,-37.372887,0,617.267514,5759.552561,1,15002.849014,857,35000,4.0,1,32.140344,0
4,9834.379665,1522.983460,1088.038782,316.104680,-45.805628,2,459.932597,28821.640152,2,4643.721244,857,35000,4.0,1,1.308767,0


In [31]:
# CHecking for negatives values
print(f"Negative closure_rate: {(df['closure_rate'] < 0).sum()}")
print(f"Negative evasion_rate: {(df['evasion_time'] < 0).sum()}")

Negative closure_rate: 0
Negative evasion_rate: 0


In [ ]:
df.missile_speed.value_counts()  # From metadata csv

missile_speed
1372    473
857     154
686     100
1715     97
2058     58
847      40
1544     39
750      20
1475     19
Name: count, dtype: int64

In [33]:
df.T

,0,1,2,3,4,5,6,7,8,9,...,990,991,992,993,994,995,996,997,998,999
launch_distance,22570.893367,2367.007196,6602.346475,18281.792405,9834.379665,12661.488033,25901.724724,8890.584630,983.734396,13859.562006,...,4981.478801,32056.695360,53419.689040,19021.028127,128594.277243,64845.033514,83532.658041,146555.411805,77971.657118,2253.861997
remaining_distance,2961.980067,1652.662358,2558.752556,18049.709650,1522.983460,3400.938832,17955.494364,7886.781225,483.793860,13391.098584,...,1605.088087,4117.674334,26058.452754,3767.541060,40310.301403,60646.537540,13843.586940,94701.450863,17925.445277,1103.757518
closure_rate,1185.763992,625.878279,871.179381,648.636953,1088.038782,725.990269,844.960185,1020.875119,1066.347972,386.272373,...,590.826598,1312.773778,1663.712597,438.758129,1394.591033,1186.846963,1211.445463,1803.707956,1636.533985,405.627102
azimuth,61.005549,212.466382,281.112640,115.135435,316.104680,259.107852,211.731931,283.740414,349.132944,176.945221,...,240.738983,191.345225,72.020487,206.765055,69.911001,251.992973,243.510094,41.013277,305.568549,117.061738
elevation,-34.707563,68.630275,45.266055,-37.372887,-45.805628,-2.007360,-83.509892,-13.091010,-43.024865,-55.574971,...,73.492841,-85.265763,-30.731328,27.430184,-76.994103,-14.818047,-43.173902,-54.366817,-32.225887,-20.136454
missile_phase,2.000000,0.000000,1.000000,0.000000,2.000000,2.000000,0.000000,0.000000,1.000000,0.000000,...,2.000000,2.000000,1.000000,2.000000,2.000000,0.000000,2.000000,1.000000,2.000000,1.000000
your_speed,825.050818,751.777144,104.527357,617.267514,459.932597,693.742453,125.237980,708.341294,291.592189,833.848608,...,685.281835,731.899168,451.183218,527.764085,292.249519,619.546047,493.576101,982.046031,537.593520,656.393231
your_altitude,6512.661972,18782.156933,3625.183143,5759.552561,28821.640152,9790.692112,1510.779084,14553.522284,4020.288191,15714.406817,...,4796.945631,4000.690960,7211.059032,2657.273049,16145.505669,23094.841893,24159.528537,26605.676886,22146.398479,1624.780195
your_maneuverability,1.000000,1.000000,2.000000,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000,0.000000,...,0.000000,2.000000,1.000000,0.000000,0.000000,1.000000,2.000000,1.000000,1.000000,0.000000
enemy_altitude,3655.798699,19508.648175,29894.396018,15002.849014,4643.721244,26557.889641,11515.140368,4119.563278,2974.936095,28617.057195,...,13657.745440,24585.421109,28243.550517,16566.700223,15476.262795,26693.600073,17534.961728,7711.296412,4263.977523,17091.003655


In [34]:
# Check feature ranges are within expected bounds
print(df[['launch_distance', 'remaining_distance', 'closure_rate', 
          'evasion_time', 'your_speed', 'your_altitude']].describe())

       launch_distance  remaining_distance  closure_rate  evasion_time  \
count      1000.000000         1000.000000   1000.000000   1000.000000   
mean      64915.002129        31821.659117   1266.905104     24.195931   
std       74767.756596        45271.208405    482.164719     31.584890   
min         669.396283            2.403273      4.027710      0.006151   
25%       10515.861139         3803.500738    872.679155      4.102251   
50%       39195.914463        14983.450620   1327.613182     11.872938   
75%       93328.236924        40924.330595   1573.072050     31.334417   
max      395478.660029       389130.451998   2858.542394    284.080232   

        your_speed  your_altitude  
count  1000.000000    1000.000000  
mean    521.111816   14957.021066  
std     263.082784    8549.393126  
min      62.358418      72.422424  
25%     297.653412    7338.047586  
50%     518.151709   15324.311391  
75%     751.001456   22210.948597  
max     982.046031   29963.768605  


In [35]:
# Check remaining_distance never exceeds launch_distance
print(f"remaining > launch: {(df['remaining_distance'] > df['launch_distance']).sum()}")

remaining > launch: 0


In [ ]:
# Check hit label distribution
print(df['hit'].value_counts())
print(df['hit'].value_counts(normalize=True))  # It shows the distribution as percentages instead of counts.

hit
0    602
1    398
Name: count, dtype: int64
hit
0    0.602
1    0.398
Name: proportion, dtype: float64


- `remaining > launch: 0` - physical constraint holds ✅
- All ranges look sensible - speeds, altitudes, distances all within expected bounds ✅
- evasion_time min is 0.006s (very close missile, terminal phase) - realistic ✅
- **Hit distribution:** 60% miss, 40% hit - reasonable balance, not skewed ✅